In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager

# Configurar Firefox en modo headless (sin abrir ventana)
firefox_options = Options()
# firefox_options.add_argument("--headless")  # Opcional, quita si quieres ver el navegador

# Inicializar el driver con WebDriver Manager
service = Service(GeckoDriverManager().install())
driver = webdriver.Firefox(service=service, options=firefox_options)


: 

In [ ]:
# URL de búsqueda avanzada en Twitter
search_url = "https://twitter.com/search?q=(from%3ACentral_CBT)%20since%3A2024-01-21%20until%3A2025-01-01&src=typed_query&f=live"
driver.get(search_url)

In [8]:

# Lista para almacenar los tweets
tweets_data = []
scroll_count = 5000  # Ajusta este número para más tweets

for _ in range(scroll_count):
    tweets = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')

    for tweet in tweets:
        try:
            text = tweet.find_element(By.XPATH, './/div[@lang]').text
            date = tweet.find_element(By.XPATH, './/time').get_attribute('datetime')
            tweets_data.append({"Fecha": date, "Texto": text})
        except:
            continue  # Evita errores si algún tweet no tiene el formato esperado

    # Hacer scroll para cargar más tweets
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Esperar que carguen nuevos tweets



KeyboardInterrupt: 

In [9]:
# Cerrar el navegador
# driver.quit()

# Guardar en CSV
df = pd.DataFrame(tweets_data)
df.to_csv("tweets_central_cbt_scraped.csv", index=False, sep=';', decimal=',')

print(f"Se han guardado {len(df)} tweets en 'tweets_central_cbt_scraped.csv'.")

Se han guardado 906 tweets en 'tweets_central_cbt_scraped.csv'.


In [ ]:
import pandas as pd
df = pd.read_csv("tweets_central_cbt_scraped.csv", sep=';', decimal=',')
df['FECHA_DIA'] = df['Fecha'].astype(str).str[:10]
df['FECHA_HORA'] = df['Fecha'].astype(str).str[11:19]
patron = r"\b(\d+-\d+-\d+)\b"
df['CODIGO'] = df['Texto'].str.extract(patron)

patron = r"\b[A-Z]+-?\d+\b"  # Para códigos tipo "B5", "B-5", "RX5", "RX-5"
df["codigo_alfanumerico"] = df["Texto"].str.findall(patron).str.join(", ")
df.to_csv("tweets_procesados.csv", index=False, sep=';', decimal=',')


,Fecha,Texto,FECHA_DIA,FECHA_HORA,CODIGO,codigo_alfanumerico
0,2025-03-19T00:22:16.000Z,10-3-4 SAN MIGUEL / 2 PONIENTE RX-5,2025-03-19,00:22:16,10-3-4,RX-5
1,2025-03-18T23:42:38.000Z,10-3-4 EJERCITO / LA MARINA RX-5,2025-03-18,23:42:38,10-3-4,RX-5
2,2025-03-18T21:22:17.000Z,"10-4-1 AVENIDA O\'HIGGINS / GRISELDA RX-5, B-4",2025-03-18,21:22:17,10-4-1,"RX-5, B-4"
3,2025-03-18T11:05:00.000Z,"10-4-1 TARAPACA / AISEN RX-5, B-8",2025-03-18,11:05:00,10-4-1,"RX-5, B-8"
4,2025-03-18T10:35:47.000Z,10-4-1 AVENIDA VASCO NÚÑEZ DE BALBOA / DETRÁS ...,2025-03-18,10:35:47,10-4-1,"RH-2, RX-5"
...,...,...,...,...,...,...
901,2023-11-11T23:08:56.000Z,10-0-1: Estamos respondiendo a una emergencia ...,2023-11-11,23:08:56,10-0-1,"F1, B5, B9"
902,2023-11-11T19:09:28.000Z,10-0-1: Estamos respondiendo a una emergencia ...,2023-11-11,19:09:28,10-0-1,"B8, B4, B5"
903,2023-11-11T00:17:26.000Z,10-3-9: Estamos respondiendo a un rescate de p...,2023-11-11,00:17:26,10-3-9,"B5, B9"
904,2023-11-10T14:16:04.000Z,10-0-1: Estamos respondiendo a un Incendio est...,2023-11-10,14:16:04,10-0-1,"Q2, B9, B5"
